# Huấn luyện 
Baseline, Decision Tree và Bagging.

Chạy sau `chuanbi_dulieu.ipynb`. Giữ đủ 12 đặc trưng, dùng chung Pipeline và 5 fold CV. Notebook này chỉ đọc train, không đánh giá test.


In [4]:
from pathlib import Path
import sys
import pandas as pd
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'data' / 'winequality-red.csv').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Đặt notebook trong thư mục notebooks của dự án.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
OUT = PROJECT_ROOT / 'output'


## 1. Cấu hình mô hình và miền siêu tham số
Sửa cấu hình chung tại `src/pipelines.py` trước khi cả hai chạy. Giữ cùng `n_iter=8`.

In [8]:
import pandas as pd
from IPython.display import display

from src.pipelines import model_pipelines, search_spaces
from src.split_experiment import GROUPS, run_group

OWNER = "vi"
models = model_pipelines()
spaces = search_spaces()

# Bảng cấu hình Pipeline
pipeline_rows = []

for name in GROUPS[OWNER]:
    steps = models[name].named_steps
    pipeline_rows.append({
        "Mô hình": name,
        "Điền khuyết": "Trung vị",
        "Chuẩn hóa": "Có" if "scaler" in steps else "Không",
        "Tìm tham số": "Có" if name in spaces else "Không"
    })

print("CẤU HÌNH PIPELINE")
display(pd.DataFrame(pipeline_rows).style.hide(axis="index"))

# Bảng miền tìm kiếm siêu tham số
param_rows = []

for name in GROUPS[OWNER]:
    for param, values in spaces.get(name, {}).items():
        param_rows.append({
            "Mô hình": name,
            "Tham số": param.removeprefix("model__"),
            "Giá trị thử": ", ".join(map(str, values))
        })

print("\nMIỀN TÌM KIẾM SIÊU THAM SỐ")
if param_rows:
    display(pd.DataFrame(param_rows).style.hide(axis="index"))

CẤU HÌNH PIPELINE


Mô hình,Điền khuyết,Chuẩn hóa,Tìm tham số
Dummy most_frequent,Trung vị,Không,Không
Dummy stratified,Trung vị,Không,Không
Logistic Regression,Trung vị,Có,Không
Decision Tree,Trung vị,Không,Có
Bagging,Trung vị,Không,Có



MIỀN TÌM KIẾM SIÊU THAM SỐ


Mô hình,Tham số,Giá trị thử
Decision Tree,max_depth,"None, 5, 10, 20"
Decision Tree,min_samples_leaf,"1, 2, 5"
Decision Tree,class_weight,"None, balanced"
Bagging,n_estimators,"100, 200"
Bagging,max_samples,"0.7, 1.0"
Bagging,estimator__max_depth,"None, 10"
Bagging,estimator__min_samples_leaf,"1, 3"


## 2. CV cấu hình ban đầu và tìm kiếm siêu tham số
Tiền xử lý được học trong mỗi fold; F1 macro là chỉ số chọn. Lưu điểm train/validation, trung bình và độ lệch chuẩn mẫu.

In [9]:
initial, tuned = run_group(PROJECT_ROOT, OWNER, n_iter=8, n_jobs=2)
display(initial)
display(tuned)

CV: vi Dummy most_frequent
CV: vi Dummy stratified
CV: vi Logistic Regression
CV: vi Decision Tree
CV: vi Bagging


,model,train_accuracy_mean,train_accuracy_std,val_accuracy_mean,val_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,train_f1_macro_mean,train_f1_macro_std,val_f1_macro_mean,val_f1_macro_std,train_f1_weighted_mean,train_f1_weighted_std,val_f1_weighted_mean,val_f1_weighted_std,fit_time_mean
0,Dummy most_frequent,0.436560,0.000147,0.436560,0.000588,0.333333,0.000000,0.333333,0.000000,0.202595,0.000048,0.202595,0.000190,0.265335,0.000152,0.265335,0.000607,0.012392
1,Dummy stratified,0.370360,0.001733,0.361366,0.020011,0.334939,0.002521,0.324063,0.021658,0.334870,0.002623,0.323582,0.022188,0.368837,0.001748,0.358968,0.020350,0.010598
2,Logistic Regression,0.592576,0.005310,0.585997,0.013996,0.549935,0.006945,0.542380,0.013247,0.561710,0.007128,0.554022,0.014397,0.587248,0.005961,0.580142,0.014278,0.056092
3,Decision Tree,1.000000,0.000000,0.526551,0.016040,1.000000,0.000000,0.513106,0.015160,1.000000,0.000000,0.512345,0.014984,1.000000,0.000000,0.527632,0.015337,0.075505
4,Bagging,1.000000,0.000000,0.614903,0.017568,1.000000,0.000000,0.584812,0.013227,1.000000,0.000000,0.595644,0.014157,1.000000,0.000000,0.612453,0.016851,4.491099


,model,stage,f1_macro,std,train_f1_macro,best_params
0,Decision Tree,tuned,0.527052,0.016549,0.775143,"{'model__min_samples_leaf': 1, 'model__max_dep..."
1,Bagging,tuned,0.601064,0.013465,0.999563,"{'model__n_estimators': 200, 'model__max_sampl..."


In [10]:
# So sánh F1 macro trước và sau tìm kiếm siêu tham số
comparison = initial[
    ["model", "train_f1_macro_mean", "val_f1_macro_mean"]
].merge(
    tuned[["model", "train_f1_macro", "f1_macro"]],
    on="model"
)

comparison.columns = [
    "Mô hình",
    "Train ban đầu",
    "CV ban đầu",
    "Train sau tinh chỉnh",
    "CV sau tinh chỉnh"
]

comparison["Thay đổi CV"] = (
    comparison["CV sau tinh chỉnh"] - comparison["CV ban đầu"]
)

display(
    comparison.style
    .hide(axis="index")
    .format(precision=4, subset=comparison.columns[1:])
)

Mô hình,Train ban đầu,CV ban đầu,Train sau tinh chỉnh,CV sau tinh chỉnh,Thay đổi CV
Decision Tree,1.0000,0.5123,0.7751,0.5271,0.0147
Bagging,1.0000,0.5956,0.9996,0.6011,0.0054
